Actions:
1. Split train/test
2. Imputasi missing value di train, transform di train dan test
3. Enoding categorical variables
4. Scale numberic features untuk Logistic Regression

In [1]:
import pandas as pd 

df = pd.read_parquet("../data/processed/credit_risk_dataset_clean.parquet", engine="pyarrow")
print(df.shape)
df.head(5)

(31677, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
5,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {y_test.shape[0]:,} samples")
print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).mul(100).round(1).to_string())

Training set: 25,341 samples
Test set: 6,336 samples

Class balance in test set:
loan_status
0    78.5
1    21.5


In [3]:
# Impute missing numeric values with median
median_rate = X_train["loan_int_rate"].median()
median_emp = X_train["person_emp_length"].median()

for d in (X_train, X_test):
    d["loan_int_rate"] = d["loan_int_rate"].fillna(median_rate)
    d["person_emp_length"] = d["person_emp_length"].fillna(median_emp)

print("Missing train:", X_train.isnull().sum().sum())
print("Missing test:", X_test.isnull().sum().sum())

Missing train: 0
Missing test: 0


In [4]:
# Encode binary categorical : Y/N -> 1/0
X_train["cb_person_default_on_file"] = (X_train["cb_person_default_on_file"] == "Y").astype(int)
X_test["cb_person_default_on_file"] = (X_test["cb_person_default_on_file"] == "Y").astype(int)

check = X_train.groupby('cb_person_default_on_file').size()
print(check)

cb_person_default_on_file
0    20845
1     4496
dtype: int64


In [5]:
# One-hot encode multi-category columns
X_train = pd.get_dummies(X_train, columns=["person_home_ownership", "loan_intent", "loan_grade"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["person_home_ownership", "loan_intent", "loan_grade"], drop_first=True)

In [6]:
X_train.head(5)

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G
15728,22,71760,6.0,15500,7.88,0.22,0,2,False,False,...,False,False,False,False,False,False,False,False,False,False
16296,24,87000,7.0,25000,10.99,0.29,0,4,False,False,...,False,False,False,False,True,False,False,False,False,False
26121,31,96000,1.0,22500,11.36,0.23,0,5,False,False,...,False,True,False,False,True,False,False,False,False,False
26653,28,110000,12.0,6000,6.03,0.05,0,5,False,False,...,False,True,False,False,False,False,False,False,False,False
1231,21,41900,2.0,16000,12.84,0.38,1,2,False,False,...,False,False,False,True,False,True,False,False,False,False


In [7]:
from sklearn.preprocessing import StandardScaler

scale_columns = ["person_age", "person_income", "person_emp_length",
                 "loan_amnt", "loan_int_rate", "loan_percent_income",
                 "cb_person_cred_hist_length"]

scaler = StandardScaler()
X_train[scale_columns] = scaler.fit_transform(X_train[scale_columns]) # fit hanya di train
X_test[scale_columns] = scaler.transform(X_test[scale_columns])       # test hanya di transform

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {y_test.shape[0]:,} samples")
print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).mul(100).round(1).to_string())

Training set: 25,341 samples
Test set: 6,336 samples

Class balance in test set:
loan_status
0    78.5
1    21.5


In [8]:
print(f'\nFinal data X_train: {X_train.shape[0]:,} rows × {X_train.shape[1]} columns')
print('✅ Preprocessing complete')


Final data X_train: 25,341 rows × 22 columns
✅ Preprocessing complete


In [9]:
X_train.to_parquet("../data/processed/X_train.parquet", engine="pyarrow", compression="snappy")
X_test.to_parquet("../data/processed/X_test.parquet", engine="pyarrow", compression="snappy")
y_train.to_frame().to_parquet("../data/processed/y_train.parquet", engine="pyarrow", compression="snappy")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet", engine="pyarrow", compression="snappy")

In [10]:
import joblib
joblib.dump(list(X_train.columns), "../models/feature_columns.pkl")

['../models/feature_columns.pkl']

In [15]:
# Simpan artefak preprocessing untuk dipakai ulang (aplikasi scoring)
joblib.dump(scaler, "../models/scaler.pkl")

preprocess_config = {
    "medians": {
        "loan_int_rate": median_rate,
        "person_emp_length": median_emp
    },
    "scale_columns": scale_columns,
    "one_hot_columns": ["person_home_ownership", "loan_intent", "loan_grade"],
    "binary_map": {"cb_person_default_on_file": {"N": 0, "Y":1}},
}
joblib.dump(preprocess_config, "../models/preprocess_config.pkl")

print("Tersimpan: scaler.pkl dan preprocess_config.pkl")
print(preprocess_config["medians"])

Tersimpan: scaler.pkl dan preprocess_config.pkl
{'loan_int_rate': np.float64(10.99), 'person_emp_length': np.float64(4.0)}


In [16]:
scaler_loaded = joblib.load("../models/scaler.pkl")
config_loaded = joblib.load("../models/preprocess_config.pkl")

print("Mean dari train:", dict(zip(scale_columns, scaler_loaded.mean_.round(2))))
print("Median:", config_loaded["medians"])

Mean dari train: {'person_age': np.float64(27.69), 'person_income': np.float64(66514.32), 'person_emp_length': np.float64(4.79), 'loan_amnt': np.float64(9650.91), 'loan_int_rate': np.float64(11.03), 'loan_percent_income': np.float64(0.17), 'cb_person_cred_hist_length': np.float64(5.78)}
Median: {'loan_int_rate': np.float64(10.99), 'person_emp_length': np.float64(4.0)}
